# YOLO: You Only Look Once

**Paper**: Redmon et al., CVPR 2016 — *You Only Look Once: Unified, Real-Time Object Detection*

## The Big Idea: Detection as Regression

R-CNN family uses **two stages**: propose regions → classify each. YOLO collapses detection into a **single network pass** — simultaneously predicting all bounding boxes and class probabilities.

<img src='../figures/yolo_v1_arch.png' width='750'/>

## How YOLO v1 Works

1. **Divide** the image into an S×S grid (S=7 for Pascal VOC)
2. Each grid cell predicts:
   - **B bounding boxes** (B=2): each box has (x, y, w, h, confidence)
   - **C class probabilities** (one per class, shared across B boxes)
3. The cell **responsible** for detecting an object is the one whose center falls in it
4. Final output tensor: S × S × (B×5 + C) = 7×7×30 for VOC (B=2, C=20)

**Confidence score** = P(object) × IoU(pred, gt) — zero if no object in cell.

## YOLO Grid: What Each Cell Predicts

<img src='../img/yolo05.png' width='400'/>

The cell containing the object center is responsible. Each cell predicts B=2 boxes plus 20 class probabilities. At test time: class-specific confidence = class prob × box confidence.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import requests
from io import BytesIO

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## YOLO v1 Network Architecture

YOLO v1 uses a **24-layer CNN** (inspired by GoogLeNet) followed by 2 FC layers. The final output is reshaped to S×S×30.

We implement a scaled-down version for demonstration — the same structure but smaller.

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, k, s, p, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.LeakyReLU(0.1, inplace=True)
        )
    def forward(self, x): return self.net(x)


class YOLOv1(nn.Module):
    """
    Simplified YOLO v1 for 448x448 input.
    Output: (B, S, S, B_boxes*5 + C)
    """
    def __init__(self, S=7, B=2, C=20):
        super().__init__()
        self.S = S; self.B = B; self.C = C

        self.backbone = nn.Sequential(
            # Block 1
            ConvBlock(3,  64, 7, 2, 3),  # 448->224
            nn.MaxPool2d(2, 2),           # 224->112
            # Block 2
            ConvBlock(64, 192, 3),
            nn.MaxPool2d(2, 2),           # 112->56
            # Block 3
            ConvBlock(192, 128, 1, 1, 0),
            ConvBlock(128, 256, 3),
            ConvBlock(256, 256, 1, 1, 0),
            ConvBlock(256, 512, 3),
            nn.MaxPool2d(2, 2),           # 56->28
            # Block 4
            ConvBlock(512, 256, 1, 1, 0),
            ConvBlock(256, 512, 3),
            ConvBlock(512, 512, 1, 1, 0),
            ConvBlock(512, 1024, 3),
            nn.MaxPool2d(2, 2),           # 28->14
            # Block 5
            ConvBlock(1024, 512, 1, 1, 0),
            ConvBlock(512, 1024, 3),
            ConvBlock(1024, 1024, 3),
            ConvBlock(1024, 1024, 3, 2, 1),  # 14->7
            ConvBlock(1024, 1024, 3),
            ConvBlock(1024, 1024, 3),
        )

        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1024 * S * S, 4096),
            nn.LeakyReLU(0.1),
            nn.Dropout(0.5),
            nn.Linear(4096, S * S * (B*5 + C)),
        )

    def forward(self, x):
        feat = self.backbone(x)
        out  = self.head(feat)
        return out.reshape(-1, self.S, self.S, self.B*5 + self.C)


model = YOLOv1(S=7, B=2, C=20)
dummy = torch.randn(1, 3, 448, 448)
out = model(dummy)
print(f'Input: {tuple(dummy.shape)}')
print(f'Output: {tuple(out.shape)}  (batch=1, S=7, S=7, B*5+C=30)')
params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {params:,}')

## YOLO v1 Loss Function

YOLO v1 uses a **sum-squared error** loss with careful weighting:

$$L = \lambda_{coord} \sum_{ij} \mathbb{1}_{ij}^{obj} [(x-\hat x)^2+(y-\hat y)^2+(\sqrt w - \sqrt{\hat w})^2+(\sqrt h - \sqrt{\hat h})^2]$$
$$+ \sum_{ij} \mathbb{1}_{ij}^{obj}(C-\hat C)^2 + \lambda_{noobj} \sum_{ij} \mathbb{1}_{ij}^{noobj}(C-\hat C)^2$$
$$+ \sum_{ij} \mathbb{1}_{ij}^{obj} \sum_c (p_c - \hat p_c)^2$$

Key choices:
- **sqrt(w), sqrt(h)**: penalizes errors more for small boxes — better small-object detection
- **lambda_coord=5**: up-weights localization loss (boxes matter more)
- **lambda_noobj=0.5**: down-weights no-object cells (they dominate the grid)

In [ ]:
class YOLOv1Loss(nn.Module):
    def __init__(self, S=7, B=2, C=20, lambda_coord=5.0, lambda_noobj=0.5):
        super().__init__()
        self.S = S; self.B = B; self.C = C
        self.lambda_coord = lambda_coord
        self.lambda_noobj = lambda_noobj

    def forward(self, pred, target):
        """
        pred, target: (N, S, S, B*5+C)
        Layout: [x,y,w,h,conf] x B, then C class probs
        """
        N = pred.shape[0]
        obj_mask  = target[..., 4] > 0   # (N, S, S) — cell has object
        noobj_mask = ~obj_mask

        # ── Box + Confidence losses for responsible predictor ──
        # For simplicity: use first box only (full YOLO picks box with higher IoU)
        pred_box  = pred[..., :4]          # x,y,w,h
        pred_conf = pred[..., 4]           # confidence
        tgt_box   = target[..., :4]
        tgt_conf  = target[..., 4]

        # Coordinate loss (only obj cells)
        box_loss_xy = F.mse_loss(pred_box[obj_mask][..., :2],
                                  tgt_box[obj_mask][..., :2], reduction='sum')
        # sqrt for w, h
        pred_wh = pred_box[obj_mask][..., 2:].clamp(min=0).sqrt()
        tgt_wh  = tgt_box[obj_mask][..., 2:].clamp(min=0).sqrt()
        box_loss_wh = F.mse_loss(pred_wh, tgt_wh, reduction='sum')

        # Confidence loss
        conf_obj_loss   = F.mse_loss(pred_conf[obj_mask],   tgt_conf[obj_mask],   reduction='sum')
        conf_noobj_loss = F.mse_loss(pred_conf[noobj_mask], tgt_conf[noobj_mask], reduction='sum')

        # Class loss (only obj cells)
        pred_cls = pred[..., self.B*5:]
        tgt_cls  = target[..., self.B*5:]
        cls_loss = F.mse_loss(pred_cls[obj_mask], tgt_cls[obj_mask], reduction='sum')

        total = (self.lambda_coord * (box_loss_xy + box_loss_wh)
                 + conf_obj_loss
                 + self.lambda_noobj * conf_noobj_loss
                 + cls_loss) / N
        return total


criterion = YOLOv1Loss()
pred_dummy = torch.randn(2, 7, 7, 30)
tgt_dummy  = torch.zeros(2, 7, 7, 30)
# Simulate one object at cell [1,2]
tgt_dummy[0, 1, 2, :5] = torch.tensor([0.5, 0.5, 0.4, 0.6, 1.0])
tgt_dummy[0, 1, 2, 10] = 1.0  # class 0 = 'aeroplane'
loss = criterion(pred_dummy, tgt_dummy)
print(f'YOLO loss: {loss.item():.4f}')

## YOLO v1 vs v2 vs v3 Evolution

| | YOLO v1 | YOLO v2 (YOLO9000) | YOLO v3 |
|---|---|---|---|
| **Year** | 2016 | 2017 | 2018 |
| **Grid** | 7×7 | 13×13 | 3 scales (13/26/52) |
| **Anchors** | None (direct pred) | 5 anchors (k-means) | 9 anchors (3 per scale) |
| **Backbone** | Custom 24-conv | Darknet-19 | Darknet-53 |
| **Multi-scale** | No | No | Yes (FPN-style) |
| **Classes** | 20 (VOC) | 9000 (WordTree) | 80 (COCO) |
| **mAP VOC07** | 63.4% | 78.6% | — |
| **Speed** | 45 fps | 67 fps | 35 fps |

**Key improvements from v1 to v3**:
- **Anchor boxes** (v2): predicts offsets relative to anchor instead of raw (x,y,w,h) — much easier to learn
- **Multi-scale detection** (v3): detects objects at 3 different grid resolutions — handles small objects better
- **Residual backbone** (v3): Darknet-53 with skip connections — much deeper without gradient issues

See `02-YOLOv3.ipynb` for the full YOLOv3 implementation with Darknet config.

## Anchor Box Predictions (YOLO v2+)

v2 introduced anchor boxes from k-means clustering of training set boxes. Instead of predicting raw (x,y,w,h), predict offsets:

$$b_x = \sigma(t_x) + c_x, \quad b_y = \sigma(t_y) + c_y$$
$$b_w = p_w e^{t_w}, \quad b_h = p_h e^{t_h}$$

where $(c_x, c_y)$ is the grid cell offset and $(p_w, p_h)$ is the anchor size.

In [ ]:
def yolo_decode_boxes(raw_preds, anchors, stride, n_classes):
    """
    Decode YOLO v2/v3 box predictions.
    raw_preds: (B, n_anchors*(5+C), H, W)
    anchors: list of (pw, ph) in pixels
    Returns decoded boxes in image coordinates.
    """
    B, _, H, W = raw_preds.shape
    n_anch = len(anchors)

    pred = raw_preds.view(B, n_anch, 5+n_classes, H, W)
    pred = pred.permute(0,1,3,4,2)  # (B, n_anch, H, W, 5+C)

    # Grid cell offsets
    grid_y = torch.arange(H, dtype=torch.float32).view(1,1,H,1)
    grid_x = torch.arange(W, dtype=torch.float32).view(1,1,1,W)

    bx = (torch.sigmoid(pred[..., 0]) + grid_x) * stride
    by = (torch.sigmoid(pred[..., 1]) + grid_y) * stride

    anchors_t = torch.tensor(anchors, dtype=torch.float32).view(1,n_anch,1,1,2)
    bw = torch.exp(pred[..., 2:3]) * anchors_t[..., 0:1]
    bh = torch.exp(pred[..., 3:4]) * anchors_t[..., 1:2]

    conf   = torch.sigmoid(pred[..., 4:5])
    cls    = torch.sigmoid(pred[..., 5:])

    return bx, by, bw, bh, conf, cls


# Demo: 3 anchors at 13x13 feature map (stride=32 for 416x416 image)
anchors_demo = [(116,90), (156,198), (373,326)]  # typical YOLO v3 large anchors
dummy_pred = torch.randn(1, 3*(5+80), 13, 13)    # COCO 80 classes
bx,by,bw,bh,conf,cls = yolo_decode_boxes(dummy_pred, anchors_demo, stride=32, n_classes=80)
print(f'Decoded box centers: bx shape={tuple(bx.shape)}')
print(f'Decoded box sizes:   bw shape={tuple(bw.shape)}')
print(f'Objectness:          conf shape={tuple(conf.shape)}')

## Summary: YOLO Family vs Two-Stage Detectors

| | Faster R-CNN | YOLO v1 | YOLO v3 |
|---|---|---|---|
| **Approach** | Two-stage | One-stage | One-stage |
| **Speed** | ~0.2 sec/img (5 fps) | 45 fps | 35 fps (better mAP) |
| **mAP VOC** | 73.2% | 63.4% | ~90%+ |
| **Small objects** | Good (RPN anchors) | Weak (1 box/cell) | Good (3 scales) |
| **Trade-off** | Accuracy | Speed | Both (but more hparam) |

YOLO's key insight: for applications where **speed matters more than per-instance precision** (autonomous driving, video analysis), a single-shot detector is the right choice.

For the full YOLOv3 implementation with Darknet config parser, see `02-YOLOv3.ipynb`.